In [ ]:
deps_path = '/kaggle/input/datasets/nhhsag12/colpali-dependency'
!pip install --no-index --find-links {deps_path} --requirement {deps_path}/requirements.txt

In [ ]:
# --- PATCH: Fix colpali_engine import conflict ---
import types, sys

def _patch_colpali_stub_all():
    problem_modules = [
        "colpali_engine.models.gemma3",
        "colpali_engine.models.gemma3.bigemma3",
        "colpali_engine.models.gemma3.colgemma3",
        "colpali_engine.models.modernvbert",
        "colpali_engine.models.modernvbert.bivbert",
        "colpali_engine.models.modernvbert.colvbert",
        "colpali_engine.models.paligemma",
        "colpali_engine.models.paligemma.bipali",
        "colpali_engine.models.paligemma.colpali",
        "colpali_engine.models.paligemma.bipali_proj",
        "colpali_engine.models.qwen2",
        "colpali_engine.models.qwen2.biqwen2",
        "colpali_engine.models.qwen2.colqwen2",
        "colpali_engine.models.qwen3",
        "colpali_engine.models.qwen3.biqwen3",
        "colpali_engine.models.qwen3.colqwen3",
        "colpali_engine.models.qwen3_5",
        "colpali_engine.models.qwen3_5.biqwen3_5",
        "colpali_engine.models.qwen3_5.colqwen3_5",
        "colpali_engine.models.qwen_omni",
        "colpali_engine.models.qwen_omni.colqwen2_5_omni",
    ]

    stub_class_map = {
        "colpali_engine.models.gemma3": [
            "BiGemma3", "BiGemmaProcessor3", "ColGemma3", "ColGemmaProcessor3",
        ],
        "colpali_engine.models.modernvbert": [
            "BiModernVBert", "BiModernVBertProcessor", "ColModernVBert", "ColModernVBertProcessor",
        ],
        "colpali_engine.models.paligemma": [
            "BiPali", "BiPaliProcessor", "BiPaliProj", "ColPali", "ColPaliProcessor",
        ],
        "colpali_engine.models.qwen2": [
            "BiQwen2", "BiQwen2Processor", "ColQwen2", "ColQwen2Processor",
        ],
        "colpali_engine.models.qwen3": [
            "BiQwen3", "BiQwen3Processor", "ColQwen3", "ColQwen3Processor",
        ],
        "colpali_engine.models.qwen3_5": [
            "BiQwen3_5", "BiQwen3_5Processor", "ColQwen3_5", "ColQwen3_5Processor",
        ],
        "colpali_engine.models.qwen_omni": [
            "ColQwen2_5Omni", "ColQwen2_5OmniProcessor",
        ],
    }

    for name in problem_modules:
        stub = types.ModuleType(name)
        sys.modules[name] = stub

    for mod_name, cls_list in stub_class_map.items():
        mod = sys.modules[mod_name]
        for cls_name in cls_list:
            setattr(mod, cls_name, type(cls_name, (), {}))

    stale = [k for k in sys.modules if k.startswith("colpali_engine") and k not in problem_modules]
    for k in stale:
        del sys.modules[k]

    print(f">>> Stubbed {len(problem_modules)} problematic submodules")

_patch_colpali_stub_all()

from colpali_engine.models.idefics3 import ColIdefics3, ColIdefics3Processor
print(f"✅ ColIdefics3 ready: {ColIdefics3}")

In [ ]:
import torch

print(torch.__version__)
print(torch.cuda.get_device_name(0))
print(torch.cuda.get_device_capability(0))

In [ ]:
# --- BƯỚC 1: SETUP DATA ---
import glob
import os
import pandas as pd
import numpy as np
import json
from tqdm.notebook import tqdm

# ==============================================================================
# CONFIG
# ==============================================================================
COLSMOL_DIR = "/kaggle/input/datasets/nguyenducdung1107/colsmol500m-layoutmmdoc/colsmol500m-pkl"
SEARCH_ROOT = "/kaggle/input"
ANNOTATIONS_PATH = "/kaggle/input/datasets/namthi/mmdocir-eval-data/MMDocIR_annotations.jsonl"
PARQUET_PATH = "/kaggle/input/datasets/namthi/mmdocir-eval-data/MMDocIR_layouts.parquet"

CURRENT_BATCH_IDX = 0 # chỉnh batch ở đây

# ==============================================================================
# STEP 1: LOAD PKL FILES & BUILD BATCH RANGES (ROBUST)
# ==============================================================================
pkl_files = sorted(glob.glob(os.path.join(COLSMOL_DIR, "*.pkl")))

print(f"Found {len(pkl_files)} PKL files")

if len(pkl_files) == 0:
    raise ValueError("❌ No PKL files found. Check COLSMOL_DIR path!")

BATCH_RANGES = []

for pkl_file in pkl_files:
    base = os.path.basename(pkl_file).replace('.pkl', '')
    
    if '-' in base:
        try:
            start, end = map(int, base.split('-'))
            BATCH_RANGES.append((start, end))
        except:
            pass

# fallback nếu parse fail
if len(BATCH_RANGES) == 0:
    print("⚠️ Using fallback batch indexing")
    BATCH_RANGES = [(i, i+1) for i in range(len(pkl_files))]

BATCH_RANGES = sorted(BATCH_RANGES)

print(f"Found {len(BATCH_RANGES)} batch ranges:")
for start, end in BATCH_RANGES[:5]:
    print(f"  [{start}:{end}]")

# validate index
if CURRENT_BATCH_IDX >= len(BATCH_RANGES):
    raise IndexError(f"❌ CURRENT_BATCH_IDX={CURRENT_BATCH_IDX} out of range")

START_IDX, END_IDX = BATCH_RANGES[CURRENT_BATCH_IDX]
print(f"\n>>> Processing batch [{START_IDX}:{END_IDX}]")

# ==============================================================================
# STEP 2: AUTO DETECT PATHS
# ==============================================================================
ENHANCED_JSONL_DIR = None
ENHANCED_IMG_DIR = None

for root, dirs, files in os.walk(SEARCH_ROOT):
    if "LAYOUT_CONTENT_FINAL" in root:
        ENHANCED_JSONL_DIR = root
    if "IMAGE ENHACED" in root or (sum(1 for f in files if f.endswith(".jpg")) > 1000):
        ENHANCED_IMG_DIR = root

# fallback
if not ENHANCED_JSONL_DIR:
    ENHANCED_JSONL_DIR = "/kaggle/input/siglip-qwen-enhaced/SIGLIP_QWEN_ENHACED/LAYOUT_CONTENT_FINAL"

if not ENHANCED_IMG_DIR:
    ENHANCED_IMG_DIR = "/kaggle/input/siglip-qwen-enhaced/SIGLIP_QWEN_ENHACED/IMAGE ENHACED"

print(f"Enhanced JSONL DIR: {ENHANCED_JSONL_DIR}")
print(f"Enhanced IMG DIR: {ENHANCED_IMG_DIR}")

# ==============================================================================
# STEP 3: BUILD DOC MAPPING (NO GLOBAL BUG)
# ==============================================================================
print("Building document mapping...")

valid_docs_in_qa = set()
with open(ANNOTATIONS_PATH, 'r') as f:
    for line in f:
        try:
            d = json.loads(line)
            valid_docs_in_qa.add(d['doc_name'].replace('.pdf', ''))
        except:
            pass

available_jsonls = glob.glob(os.path.join(ENHANCED_JSONL_DIR, "*.jsonl"))

jsonl_map = {}
for p in available_jsonls:
    fname = os.path.basename(p).replace('_layout.jsonl', '')
    jsonl_map[fname] = p

intersection_docs = sorted(list(valid_docs_in_qa.intersection(jsonl_map.keys())))
print(f"→ Found {len(intersection_docs)} valid documents")

# ==============================================================================
# STEP 4: SELECT BATCH DOCS
# ==============================================================================
target_doc_names = intersection_docs[START_IDX:END_IDX]
target_files = [jsonl_map[d] for d in target_doc_names if d in jsonl_map]

print(f"Processing batch [{START_IDX}:{END_IDX}]")
print(f"→ Docs: {len(target_doc_names)}")

if len(target_doc_names) == 0:
    raise ValueError("❌ Batch is empty! Check batch index or data.")

# ==============================================================================
# STEP 5: LOAD DATA
# ==============================================================================
print("Loading Parquet...")
df_orig = pd.read_parquet(PARQUET_PATH)
df_orig['join_doc_name'] = df_orig['doc_name'].str.replace('.pdf', '', regex=False)
df_orig = df_orig[df_orig['join_doc_name'].isin(target_doc_names)]

print("Loading JSONL enrichment...")
dfs = []

for f in tqdm(target_files, desc="Reading JSONLs"):
    try:
        temp = pd.read_json(f, lines=True)
        temp['join_doc_name'] = os.path.basename(f).replace('_layout.jsonl', '')
        
        if 'layout' in temp.columns:
            temp = temp.rename(columns={'layout': 'layout_id'})
        
        cols = ['join_doc_name', 'layout_id', 'vlm_text', 'img_enhanced_path']
        if 'text_level' in temp.columns:
            cols.append('text_level')
        
        temp = temp[[c for c in cols if c in temp.columns]]
        dfs.append(temp)
    except Exception as e:
        print(f"Skip {f}: {e}")

if len(dfs) > 0:
    df_enh = pd.concat(dfs, ignore_index=True)
    df_enh = df_enh.rename(columns={
        'vlm_text': 'vlm_text_enhanced',
        'text_level': 'text_level_enhanced'
    })
else:
    df_enh = pd.DataFrame()

# ==============================================================================
# STEP 6: MERGE
# ==============================================================================
print("Merging data...")
df_final = pd.merge(df_orig, df_enh, on=['join_doc_name', 'layout_id'], how='left')
df_final = df_final.sort_values(by=['join_doc_name', 'page_id', 'layout_id'])

# ==============================================================================
# STEP 7: CONTEXT BUILDING
# ==============================================================================
def identify_header(row):
    if row.get('type') in ['title', 'section_header', 'header']:
        return str(row.get('text', ''))
    if pd.notna(row.get('text_level_enhanced')):
        return str(row.get('text', ''))
    return np.nan

df_final['temp_header'] = df_final.apply(identify_header, axis=1)
df_final['current_section'] = (
    df_final.groupby('join_doc_name')['temp_header']
    .ffill()
    .fillna("General Content")
)

# ==============================================================================
# STEP 8: IMAGE MAP
# ==============================================================================
enh_image_map = {}

if os.path.exists(ENHANCED_IMG_DIR):
    for f in glob.glob(os.path.join(ENHANCED_IMG_DIR, "*")):
        enh_image_map[os.path.basename(f)] = f

# ==============================================================================
# STEP 9: FINAL SOURCE SELECTION
# ==============================================================================
def get_best_sources(row):
    img_type, img_data = None, None

    # image priority
    if pd.notna(row.get('img_enhanced_path')):
        fname = os.path.basename(str(row['img_enhanced_path']))
        if fname in enh_image_map:
            img_type, img_data = 'path', enh_image_map[fname]

    if img_data is None and pd.notna(row.get('image_binary')):
        img_type, img_data = 'binary', row['image_binary']

    # text priority
    raw_content = ""

    if pd.notna(row.get('vlm_text_enhanced')) and len(str(row['vlm_text_enhanced'])) > 5:
        raw_content = str(row['vlm_text_enhanced'])
    elif pd.notna(row.get('text')) and len(str(row['text'])) > 5:
        raw_content = str(row['text'])
    elif pd.notna(row.get('ocr_text')) and len(str(row['ocr_text'])) > 5:
        raw_content = str(row['ocr_text'])
    elif pd.notna(row.get('vlm_text')):
        raw_content = str(row['vlm_text'])

    section = row.get('current_section', '')
    final_text_prompt = f"Section: {section}\nContent: {raw_content}"

    if len(final_text_prompt) < 10:
        final_text_prompt = "Document layout."

    return pd.Series(
        [img_type, img_data, final_text_prompt],
        index=['img_type', 'img_data', 'final_text']
    )

print("Building final dataset...")
processed = df_final.apply(get_best_sources, axis=1)

sample_layouts_df = (
    pd.concat([df_final, processed], axis=1)
    .dropna(subset=['img_type'])
    .reset_index(drop=True)
)

# ==============================================================================
# DONE
# ==============================================================================
print("=" * 60)
print(f"✅ BATCH [{START_IDX}-{END_IDX}] READY!")
print(f"Total Layouts: {len(sample_layouts_df)}")
print("=" * 60)

In [ ]:
# --- BƯỚC 2: LOAD MODEL + LORA ---
print(">>> Loading ColSmolVLM-500M + LoRA...")

import torch
import gc
from peft import PeftModel
from colpali_engine.models import ColIdefics3, ColIdefics3Processor

gc.collect()
torch.cuda.empty_cache()

device = "cuda" if torch.cuda.is_available() else "cpu"

BASE_MODEL = "/kaggle/input/datasets/nguyenducdung1107/model500m"
LORA_PATH = "/kaggle/input/datasets/nguyenducdung1107/colsmol500-adapter/colsmol500_adapter"  # <-- đường dẫn adapter của m

# 🔹 Load base model
model = ColIdefics3.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation="eager"
)

# 🔹 Load LoRA adapter
model = PeftModel.from_pretrained(
    model,
    LORA_PATH
)

model.eval()

# 🔹 Processor
processor = ColIdefics3Processor.from_pretrained(BASE_MODEL)

print(" Model + LoRA Ready!")

In [ ]:
query_model=model
query_processor=processor

In [ ]:
print(">>> CELL 1: Infrastructure loading ...")

import torch
import torch.nn.functional as F
import numpy as np
import json, os, pickle, gc, glob, time
import pandas as pd
from tqdm.notebook import tqdm

# ==============================================================================
# CONFIG TĨNH — paths, data, batch sizes (không phải method knob)
# ==============================================================================
QUERY_BATCH_SIZE    = 50
ADC_DOC_CHUNK       = 4000
WALLCLOCK_N_WARMUP  = 5
WALLCLOCK_N_MEASURE = 50
TOPK_RATIOS         = [0.3, 0.5, 0.7, 1.0]   # query-side ablation baselines

WORKING_DIR        = "/kaggle/working"
ANNOTATIONS_PATH   = "/kaggle/input/datasets/namthi/mmdocir-eval-data/MMDocIR_annotations.jsonl"
COLSMOL_DIR        = "/kaggle/input/datasets/nguyenducdung1107/colsmol500m-layoutmmdoc/colsmol500m-pkl"
ENHANCED_JSONL_DIR = "/kaggle/input/datasets/cdnghnam/siglip-qwen-enhaced/SIGLIP_QWEN_ENHACED/LAYOUT_CONTENT_FINAL"
ENHANCED_IMG_DIR   = "/kaggle/input/datasets/cdnghnam/siglip-qwen-enhaced/SIGLIP_QWEN_ENHACED/IMAGE ENHACED"
PARQUET_PATH       = "/kaggle/input/datasets/namthi/mmdocir-eval-data/MMDocIR_layouts.parquet"
BATCH_RANGE_PKL_OVERRIDE = {
    (0, 25): "/kaggle/input/datasets/nguyenducdung1107/colsmol500m-layoutmmdoc/colsmol500m-pkl/0-25.pkl",
}

device = "cuda" if torch.cuda.is_available() else "cpu"

# ==============================================================================
# MODEL ACCESSOR
# ==============================================================================
def get_model_and_processor():
    missing = [v for v in ('query_model', 'query_processor') if v not in globals()]
    if missing:
        raise RuntimeError(f"Variables {missing} not found. Run model load cell first.")
    return globals()['query_model'], globals()['query_processor']

try:
    _m, _p = get_model_and_processor()
    print(f"✅ Model ready: {type(_m).__name__}")
    del _m, _p
except RuntimeError as _e:
    print(f"❌ {_e}"); raise

# ==============================================================================
# TRACKERS
# ==============================================================================
class TFLOPsTracker:
    def __init__(self): self._d = {}
    def add(self, key, subkey, flops):
        self._d.setdefault(key, {}).setdefault(subkey, []).append(flops / 1e12)
    def mean(self, key, subkey='default'):
        v = self._d.get(key, {}).get(subkey, [])
        return float(np.mean(v)) if v else float('nan')
    def to_df(self):
        rows = []
        for m, rd in self._d.items():
            for sk, v in rd.items():
                rows.append({'method': m, 'subkey': sk,
                             'avg_tflops': round(float(np.mean(v)), 8), 'n': len(v)})
        return pd.DataFrame(rows).sort_values(['method', 'subkey']).reset_index(drop=True)


class WallClockTracker:
    def __init__(self): self._times = {}
    def record(self, key, ms):
        self._times.setdefault(key, []).append(ms)
    def stats(self, key):
        v = self._times.get(key, [])
        if not v:
            return {'median_ms': float('nan'), 'mean_ms': float('nan'),
                    'std_ms': float('nan'), 'n': 0}
        return {'median_ms': float(np.median(v)), 'mean_ms': float(np.mean(v)),
                'std_ms': float(np.std(v)), 'n': len(v)}
    def to_df(self):
        rows = []
        for k, v in self._times.items():
            if not v: continue
            rows.append({'method': k,
                         'median_ms': round(float(np.median(v)), 4),
                         'mean_ms':   round(float(np.mean(v)),   4),
                         'std_ms':    round(float(np.std(v)),    4),
                         'n':         len(v)})
        if not rows:
            return pd.DataFrame(columns=['method', 'median_ms', 'mean_ms', 'std_ms', 'n'])
        return pd.DataFrame(rows).sort_values('median_ms').reset_index(drop=True)


class MemoryTracker:
    def __init__(self): self._entries = {}
    def record_tensor(self, key, *tensors):
        total = sum(t.numel() * t.element_size()
                    for t in tensors if isinstance(t, torch.Tensor))
        self._entries[key] = total / (1024 ** 2)
    def record_mb(self, key, mb): self._entries[key] = mb
    def get(self, key): return self._entries.get(key, float('nan'))
    def to_df(self):
        return pd.DataFrame([{'method': k, 'memory_mb': round(v, 2)}
                              for k, v in self._entries.items()]
                             ).sort_values('memory_mb').reset_index(drop=True)


tflops_tracker = TFLOPsTracker()
wallclock      = WallClockTracker()
mem_tracker    = MemoryTracker()

# ==============================================================================
# CUDA TIMING UTIL
# ==============================================================================
def cuda_time_ms(fn, device, n_warmup=0):
    for _ in range(n_warmup): fn()
    is_cuda = (str(device) in ('cuda', 'cuda:0') or
               (isinstance(device, torch.device) and device.type == 'cuda'))
    if is_cuda:
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        result = fn()
        torch.cuda.synchronize()
    else:
        t0 = time.perf_counter()
        result = fn()
    return (time.perf_counter() - t0) * 1000.0, result

# ==============================================================================
# FLOP COUNTERS
# ==============================================================================
def flops_maxsim(Q, N, L, D): return 2.0 * Q * N * L * D
def flops_adc_rvq(Q, N, L, S, K, D): return float(S * (2.0 * Q * K * D + Q * N * L))

# ==============================================================================
# SCORING UTILS  — không phụ thuộc method
# ==============================================================================
def uniform_maxsim_scores(q_norm, doc_matrix, doc_mask, chunk_size=ADC_DOC_CHUNK):
    """Standard MaxSim: sum over query tokens of max-over-doc-tokens similarity."""
    Q   = q_norm.shape[0]
    N   = doc_matrix.shape[0]
    dev = q_norm.device
    out = torch.zeros(N, device=dev)
    for s in range(0, N, chunk_size):
        e   = min(s + chunk_size, N)
        sim = torch.einsum('qd,nld->qnl', q_norm.float(), doc_matrix[s:e].float())
        sim.masked_fill_(~doc_mask[s:e].unsqueeze(0), float('-inf'))
        ms  = sim.max(dim=-1).values
        ms  = ms.masked_fill(ms == float('-inf'), 0.0)
        out[s:e] = ms.sum(0)
    return out


def topk_query_tokens(q_norm, magnitude, ratio):
    """Query-side top-k pruning baseline — selects top-ratio fraction by magnitude."""
    Q = q_norm.shape[0]
    k = max(1, int(Q * ratio))
    if k >= Q:
        return q_norm, torch.arange(Q, device=q_norm.device)
    topk_idx = magnitude.topk(k).indices.sort().values
    return q_norm[topk_idx], topk_idx

# ==============================================================================
# [P1] QUERY ENCODING
# ==============================================================================
def build_content_mask(inputs, processor):
    """Mask out special tokens (pad/bos/eos/...) — tokenizer-only logic."""
    attn_mask = inputs["attention_mask"]
    input_ids = inputs.get("input_ids", None)
    if input_ids is None:
        return attn_mask.float()
    tok = getattr(processor, 'tokenizer', processor)
    special_ids = set()
    for attr in ['pad_token_id', 'bos_token_id', 'eos_token_id',
                 'unk_token_id', 'sep_token_id', 'cls_token_id']:
        tid = getattr(tok, attr, None)
        if tid is not None: special_ids.add(int(tid))
    if hasattr(tok, 'added_tokens_encoder'):
        for _, tid in tok.added_tokens_encoder.items():
            special_ids.add(int(tid))
    if not special_ids:
        return attn_mask.float()
    special_tensor = torch.tensor(list(special_ids), device=input_ids.device)
    is_special = (input_ids.unsqueeze(-1) == special_tensor).any(dim=-1)
    return attn_mask.float() * (~is_special).float()


def encode_all_queries(qa_pairs, processor, model, device):
    """
    Encode all queries to normalized token embeddings + magnitude.
    Returns list of dicts: q_norm, magnitude, Mc, gt_set, lmr, domain, doc_name, question, gq
    """
    encoded = []
    for i in range(0, len(qa_pairs), QUERY_BATCH_SIZE):
        batch = qa_pairs[i:i + QUERY_BATCH_SIZE]
        q_in  = processor.process_queries([it['question'] for it in batch]).to(device)
        with torch.no_grad():
            q_out = model(**{k: v for k, v in q_in.items()})
            q_embs = q_out.float() if isinstance(q_out, torch.Tensor) \
                     else q_out.last_hidden_state.float()
        cmasks = build_content_mask(q_in, processor).float()
        for j, item in enumerate(batch):
            cidx        = torch.where(cmasks[j] > 0)[0]
            raw_content = q_embs[j][cidx]
            magnitude   = raw_content.norm(dim=-1)
            q_norm      = F.normalize(raw_content, dim=-1)
            encoded.append({
                'q_norm':    q_norm.cpu(),
                'magnitude': magnitude.cpu(),
                'Mc':        cidx.numel(),
                'gt_set':    item['gt_pos_set'],
                'lmr':       item['layout_mapping_raw'],
                'domain':    item['domain'],
                'doc_name':  item['doc_name'],
                'question':  item['question'],
                'gq':        i + j,
            })
        del q_in, q_embs, cmasks
    return encoded

# ==============================================================================
# [P2] DOC MATRIX BUILDER
# ==============================================================================
def _coerce(item):
    if isinstance(item, torch.Tensor): return item
    if isinstance(item, dict):
        for k in ('embedding', 'embeddings', 'vector', 'vectors', 'token_embeddings'):
            if k in item and isinstance(item[k], torch.Tensor): return item[k]
        tvs = [(k, v) for k, v in item.items() if isinstance(v, torch.Tensor)]
        if tvs: return tvs[0][1]
    raise ValueError(f"Cannot coerce type: {type(item)}")


def build_full(docs_list, device):
    """Build padded + normalized doc matrix (float16 on CUDA). Unchanged across methods."""
    ts = [_coerce(x) for x in docs_list]
    ts = [t.squeeze(0) if t.dim() > 2 else (t.unsqueeze(0) if t.dim() == 1 else t)
          for t in ts]
    N    = len(ts)
    Lmax = max(t.shape[0] for t in ts)
    D    = ts[0].shape[1]
    dtype = torch.float16 if torch.device(device).type == 'cuda' else torch.float32
    pad  = torch.zeros(N, Lmax, D, device=device, dtype=dtype)
    mask = torch.zeros(N, Lmax, device=device, dtype=torch.bool)
    for i, t in enumerate(ts):
        L = t.shape[0]
        pad[i, :L]  = F.normalize(t.float().to(device), dim=-1).to(dtype)
        mask[i, :L] = True
    return pad, mask

# ==============================================================================
# METRIC HELPERS
# ==============================================================================
def _parse_bbox(raw):
    if raw is None: return None
    try:
        if isinstance(raw, np.ndarray):
            f = raw.flatten()
            return [float(x) for x in f] if len(f) == 4 else None
    except: pass
    if isinstance(raw, (list, tuple)) and len(raw) == 4:
        try: return [float(x) for x in raw]
        except: return None
    if isinstance(raw, dict):
        for keys in [('x1','y1','x2','y2'), ('top','left','bottom','right'),
                     ('left','top','right','bottom')]:
            if all(k in raw for k in keys):
                try: return [float(raw[k]) for k in keys]
                except: pass
    return None


def overlap_area(b1, b2):
    it = max(b1[0], b2[0]); il = max(b1[1], b2[1])
    ib = min(b1[2], b2[2]); ir = min(b1[3], b2[3])
    return (ib - it) * (ir - il) if it < ib and il < ir else 0.0


def recall_area(top_k, bbox_list, lmr):
    ra = 0.0
    for p in top_k:
        info = bbox_list[p] if 0 <= p < len(bbox_list) else None
        if info is None: continue
        pg, t, l, b, r = info
        for gt in lmr:
            if pg != gt["page"]: continue
            gb = _parse_bbox(gt.get("bbox"))
            if gb: ra += overlap_area([t, l, b, r], gb)
    ga = 0.0
    for gt in lmr:
        gb = _parse_bbox(gt.get("bbox"))
        if gb: t2, l2, b2, r2 = gb; ga += max(0, b2-t2) * max(0, r2-l2)
    return 0.0 if ga <= 0 else min(ra / ga, 1.0)


def ndcg(ranked, gt, k):
    dcg  = sum(1 / np.log2(r + 2) for r, i in enumerate(ranked[:k]) if i in gt)
    idcg = sum(1 / np.log2(r + 2) for r in range(min(len(gt), k)))
    return dcg / idcg if idcg > 0 else 0.0


def hit_metrics(top10, gt_set, bbox_list, lmr):
    if not lmr: return None
    h = next((r + 1 for r, i in enumerate(top10) if i in gt_set), -1)
    return {
        'r1':       int(h != -1 and h <= 1),
        'r5':       int(h != -1 and h <= 5),
        'r10':      int(h != -1 and h <= 10),
        'recall1':  recall_area(top10[:1],  bbox_list, lmr),
        'recall5':  recall_area(top10[:5],  bbox_list, lmr),
        'recall10': recall_area(top10[:10], bbox_list, lmr),
        'n1':       ndcg(top10, gt_set, 1),
        'n5':       ndcg(top10, gt_set, 5),
        'n10':      ndcg(top10, gt_set, 10),
    }

# ==============================================================================
# METRIC STORE
# ==============================================================================
def _init_m():
    return {'r1': 0, 'r5': 0, 'r10': 0, 'n1': 0., 'n5': 0., 'n10': 0.,
            'recall1': 0., 'recall5': 0., 'recall10': 0., 'count': 0}

def _add(d, s):
    for f in ('r1', 'r5', 'r10'): d[f] += int(s[f])
    for f in ('n1', 'n5', 'n10', 'recall1', 'recall5', 'recall10'): d[f] += float(s[f])
    d['count'] += 1

def _ens(store, k):
    if k not in store: store[k] = _init_m()
    return store[k]

all_metrics        = {}
all_domain_metrics = {}
all_query_results  = []
all_batch_stats    = []

def record(key, m, domain):
    _add(_ens(all_metrics,  key), m)
    if domain not in all_domain_metrics: all_domain_metrics[domain] = {}
    _add(_ens(all_domain_metrics[domain], key), m)

# ==============================================================================
# LOAD INDEX FILES & BUILD BATCH LIST
# ==============================================================================
pkl_files = sorted(glob.glob(os.path.join(COLSMOL_DIR, "*.pkl")))
print(f"Found {len(pkl_files)} PKL files")

BATCH_RANGES = []
for p in pkl_files:
    base = os.path.basename(p).replace('.pkl', '')
    try:
        s, e = map(int, base.split(' ')[-1].split('-'))
        BATCH_RANGES.append((s, e))
    except: pass
BATCH_RANGES = sorted(BATCH_RANGES) or [(i, i+1) for i in range(len(pkl_files))]
for r in list(BATCH_RANGE_PKL_OVERRIDE.keys()):
    if r not in BATCH_RANGES:
        BATCH_RANGES = sorted(set(BATCH_RANGES + [r]))

valid_docs = set()
with open(ANNOTATIONS_PATH) as f:
    for line in f:
        try: valid_docs.add(json.loads(line)['doc_name'].replace('.pdf', ''))
        except: pass

jsonl_map = {
    os.path.basename(p).replace('_layout.jsonl', ''): p
    for p in glob.glob(os.path.join(ENHANCED_JSONL_DIR, "*.jsonl"))
}
intersection_docs = sorted(valid_docs.intersection(jsonl_map.keys()))
print(f"Intersection docs: {len(intersection_docs)}")
print(f"Batch ranges     : {BATCH_RANGES}")
print("\n>>> CELL 1 DONE — run Cell 2 to train quantizers and evaluate")

In [ ]:
print(">>> CELL 1: Load Training Corpus → Embed → Train RVQ (KMeans + SPN) → Save 2 Codebooks")
print(">>> Offline training — runs ONCE, zero test-set leakage")

import gc, os, io, time, pickle, random, subprocess, sys, types
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from tqdm.notebook import tqdm

# ── Install vector-quantize-pytorch ──────────────────────────────────────────
_rvq_wheel_dir = "/kaggle/input/datasets/thinam4/rvq-wheels/rvq_wheels"
if os.path.isdir(_rvq_wheel_dir):
    subprocess.run(
        ["pip", "install", "--quiet", "--no-index",
         "--find-links", _rvq_wheel_dir, "vector-quantize-pytorch"],
        check=True
    )
from vector_quantize_pytorch import ResidualVQ

# ==============================================================================
# CONFIG
# ==============================================================================
TRAIN_PARQUET_DIR = "/kaggle/input/datasets/nguyenducdung1107/training-dataset-mmdocir/MMDOC_TRAINING/parquet"
BASE_MODEL        = "/kaggle/input/datasets/nguyenducdung1107/model500m"
LORA_PATH         = "/kaggle/input/datasets/nguyenducdung1107/colsmol500-adapter/colsmol500_adapter"
WORKING_DIR       = "/kaggle/working"

# ── 2 Output files ────────────────────────────────────────────────────────────
CODEBOOK_SAVE_KMEANS = os.path.join(WORKING_DIR, "rvq_codebooks_kmeans.pt")
CODEBOOK_SAVE_SPN    = os.path.join(WORKING_DIR, "rvq_codebooks_spn.pt")

# ── Dataset selection ─────────────────────────────────────────────────────────
ALL_DATASETS  = ["SlideVQA", "ArxivQA", "MP-DocVQA", "TAT-DQA",
                 "SciQAG", "Wiki-ss", "DUDE"]
SAMPLE_FRAC   = 0.5
MAX_PAGES     = 80_000
RANDOM_SEED   = 42
EMBED_BATCH   = 16

# ── RVQ architecture ──────────────────────────────────────────────────────────
RVQ_N_QUANTIZERS_COARSE = 8
RVQ_N_QUANTIZERS_FINE   = 16
RVQ_CODEBOOK_SIZE       = 256

# ── RVQ Training hyper-params ─────────────────────────────────────────────────
RVQ_TRAINING_EPOCHS     = 120
RVQ_EARLY_STOP_PATIENCE = 20
RVQ_TRAINING_BATCH_SIZE = 4096
RVQ_TRAINING_SAMPLES    = 200_000
RVQ_DROPOUT_P           = 0.5
RVQ_DROPOUT_WARMUP_EPS  = 10

# ── Norm filter ───────────────────────────────────────────────────────────────
TOP_K_NORM_FRAC = 0.5

# ── SPN hyper-params ──────────────────────────────────────────────────────────
SPN_EPOCHS         = 30
SPN_BATCH_SIZE     = 2048
SPN_LR             = 3e-3
SPN_HIDDEN_DIM     = 512
SPN_TEMP_START     = 1.0
SPN_TEMP_END       = 0.1
SPN_ENTROPY_WEIGHT = 0.5

# ── Ranking loss hyper-params ─────────────────────────────────────────────────
# Contrastive (triplet margin) loss weight and margin
RANK_LOSS_WEIGHT   = 1.0          # λ_rank in L = L_compress + λ_rank * L_rank
RANK_MARGIN        = 0.2          # margin γ in max(0, γ - MaxSim(q,pos) + MaxSim(q,neg))
RANK_NEG_SAMPLES   = 8            # number of in-batch negatives per anchor

# OT (Wasserstein) loss hyper-params
OT_LOSS_WEIGHT     = 0.3          # λ_OT in total loss
OT_SINKHORN_ITERS  = 20           # Sinkhorn iterations for entropic OT
OT_SINKHORN_EPS    = 0.05         # entropic regularisation ε (smaller = closer to exact OT)

_EMBED_KEYS = ['embedding', 'embeddings', 'patches', 'tokens', 'features', 'data']

META_COLS = ["file_name", "page"]

# ==============================================================================
# STEP 2: LOAD + SAMPLE TRAINING PARQUET (lazy, low-mem)
# ==============================================================================
print("\n" + "="*70)
print("STEP 2: Load training parquet (lazy sampling)")
print("="*70)

def _sample_parquet_meta(path, dataset_name, frac, seed, max_rows=None):
    pf = pq.ParquetFile(path)
    total = pf.metadata.num_rows
    target = max(1, int(total * frac))
    if max_rows: target = min(target, max_rows)

    rng = random.Random(seed)
    group_ids = list(range(pf.num_row_groups))
    rng.shuffle(group_ids)

    frames = []; selected = 0
    for rg in group_ids:
        if selected >= target: break
        try:
            table = pf.read_row_group(rg, columns=META_COLS)
        except Exception as e:
            print(f"  [WARN] skip rg={rg}: {e}"); continue
        part = table.to_pandas().reset_index(drop=True); del table
        part["row_group_id"] = rg
        part["row_in_group"]  = range(len(part))
        left = target - selected
        if len(part) > left:
            part = part.sample(n=left, random_state=seed + rg).reset_index(drop=True)
        part["dataset"]        = dataset_name
        part["source_parquet"] = path
        frames.append(part); selected += len(part)
        del part; gc.collect()

    if not frames:
        return pd.DataFrame(), total
    return pd.concat(frames, ignore_index=True), total

dfs = []
for name in ALL_DATASETS:
    path = os.path.join(TRAIN_PARQUET_DIR, f"{name}_filter.parquet")
    if not os.path.exists(path):
        print(f"  [SKIP] {name}: not found"); continue
    try:
        df_tmp, n_full = _sample_parquet_meta(path, name, SAMPLE_FRAC, RANDOM_SEED)
        dfs.append(df_tmp)
        print(f"  {name}: {n_full} rows → sampled {len(df_tmp)}")
        del df_tmp; gc.collect()
    except Exception as e:
        print(f"  [ERROR] {name}: {e}")

if not dfs:
    raise RuntimeError("No dataset loaded. Check parquet paths.")

layouts_df = pd.concat(dfs, ignore_index=True).reset_index(drop=True)
del dfs; gc.collect()

if MAX_PAGES and len(layouts_df) > MAX_PAGES:
    layouts_df = layouts_df.sample(n=MAX_PAGES, random_state=RANDOM_SEED).reset_index(drop=True)
    print(f"  Capped to {MAX_PAGES} pages")

layouts_df["join_doc_name"] = (
    layouts_df["dataset"] + "__" + layouts_df["file_name"].astype(str))
layouts_df = layouts_df.sort_values(
    ["source_parquet", "row_group_id", "row_in_group"]).reset_index(drop=True)

print(f"\n  Total pages: {len(layouts_df)}")
print(f"  Dataset distribution:\n{layouts_df['dataset'].value_counts().to_string()}")

# ==============================================================================
# STEP 3: EMBED TRAINING PAGES
# ==============================================================================
print("\n" + "="*70)
print("STEP 3: Embed training pages with ColSmolVLM")
print("="*70)

def _build_text_from_layouts(layouts):
    if not layouts: return "Document page."
    text_keys = ["text", "ocr_text", "content", "caption", "value"]
    texts = []
    for lay in layouts:
        if not isinstance(lay, dict): continue
        for tk in text_keys:
            v = lay.get(tk)
            if v and isinstance(v, str) and len(v.strip()) > 2:
                texts.append(v.strip()); break
    return " ".join(texts)[:1000] if texts else "Document page."

def embed_pages_from_parquet(layouts_df, model, proc, dev, batch_size=EMBED_BATCH):
    n = len(layouts_df)
    parquet_cache = {}
    fused_index   = [None] * n

    def _get_pf(path):
        if path not in parquet_cache:
            parquet_cache[path] = pq.ParquetFile(path)
        return parquet_cache[path]

    batch_imgs, batch_texts, batch_pos = [], [], []
    grouped = layouts_df.groupby(["source_parquet", "row_group_id"], sort=False)

    for (source_path, rg), group_df in tqdm(
            grouped, total=grouped.ngroups, desc="  Row groups"):
        try:
            pf    = _get_pf(source_path)
            table = pf.read_row_group(int(rg), columns=["layouts", "image"])
            rg_data = table.to_pydict(); del table
        except Exception as e:
            print(f"  [WARN] read error ({source_path}, rg={rg}): {e}"); continue

        layouts_col = rg_data.get("layouts", [])
        image_col   = rg_data.get("image",   [])

        for _, row in group_df.iterrows():
            rig = int(row["row_in_group"])
            pos = int(row.name)

            try:
                layouts = layouts_col[rig] if rig < len(layouts_col) else None
                img_obj = image_col[rig]   if rig < len(image_col)   else None
                raw = (img_obj if isinstance(img_obj, bytes)
                       else img_obj.get("bytes") if isinstance(img_obj, dict)
                       else None)
                img  = (Image.open(io.BytesIO(raw)).convert("RGB")
                        if raw and len(raw) > 0
                        else Image.new("RGB", (224, 224), "white"))
                if img.width < 14 or img.height < 14:
                    img = Image.new("RGB", (224, 224), "white")
                text = _build_text_from_layouts(layouts)
            except Exception:
                img  = Image.new("RGB", (224, 224), "white")
                text = "Document page."

            batch_imgs.append(img)
            batch_texts.append(str(text).replace("<image>", " ")[:1000])
            batch_pos.append(pos)

            if len(batch_imgs) >= batch_size:
                with torch.no_grad():
                    try:
                        vis_inp = proc.process_images(batch_imgs).to(dev)
                        out_vis = model(**vis_inp)
                        txt_inp = proc.process_queries(
                            batch_texts, max_length=192, suffix="").to(dev)
                        out_txt = model(**txt_inp)
                        for k in range(len(batch_imgs)):
                            vis_e = out_vis[k].cpu().float().numpy()
                            txt_e = out_txt[k].cpu().float().numpy()
                            fused = np.concatenate([vis_e, txt_e], axis=0).astype(np.float16)
                            fused_index[batch_pos[k]] = fused
                        del vis_inp, txt_inp, out_vis, out_txt
                    except Exception as e:
                        print(f"  [WARN] embed error: {e}")
                batch_imgs.clear(); batch_texts.clear(); batch_pos.clear()
                torch.cuda.empty_cache(); gc.collect()

        del rg_data; gc.collect()

    if batch_imgs:
        with torch.no_grad():
            try:
                vis_inp = proc.process_images(batch_imgs).to(dev)
                out_vis = model(**vis_inp)
                txt_inp = proc.process_queries(
                    batch_texts, max_length=192, suffix="").to(dev)
                out_txt = model(**txt_inp)
                for k in range(len(batch_imgs)):
                    vis_e = out_vis[k].cpu().float().numpy()
                    txt_e = out_txt[k].cpu().float().numpy()
                    fused = np.concatenate([vis_e, txt_e], axis=0).astype(np.float16)
                    fused_index[batch_pos[k]] = fused
            except Exception as e:
                print(f"  [WARN] final batch error: {e}")
        torch.cuda.empty_cache(); gc.collect()

    parquet_cache.clear()
    fused_index = [e for e in fused_index if e is not None]
    return fused_index

t0 = time.perf_counter()
fused_train = embed_pages_from_parquet(
    layouts_df, model, processor, device, EMBED_BATCH)
print(f"\n  Embedded {len(fused_train)} pages in {time.perf_counter()-t0:.1f}s")

if len(fused_train) == 0:
    raise RuntimeError("No pages embedded. Check parquet + model paths.")

emb_dim = fused_train[0].shape[1]
print(f"  Embedding dim per patch: {emb_dim}")

del model, processor
gc.collect(); torch.cuda.empty_cache()
print("  Model freed from VRAM")

# ==============================================================================
# STEP 4: NORM FILTER + PATCH SAMPLING
# ==============================================================================
print("\n" + "="*70)
print("STEP 4: Norm-filter patches + sample training pool")
print("="*70)

def top_norm_filter(arr: np.ndarray, frac: float = TOP_K_NORM_FRAC) -> np.ndarray:
    """Keep top-frac patches by L2 norm. Removes whitespace/background noise."""
    L = arr.shape[0]
    if L <= 1 or frac >= 1.0:
        return arr
    k = max(1, int(L * frac))
    norms   = np.linalg.norm(arr.astype(np.float32), axis=-1)
    top_idx = np.argpartition(norms, -k)[-k:]
    return arr[np.sort(top_idx)]

def sample_patches(emb_list, n_samples: int, seed: int = 42) -> torch.Tensor:
    rng      = np.random.default_rng(seed)
    filtered = [top_norm_filter(e.astype(np.float32)) for e in emb_list]
    all_lens = [a.shape[0] for a in filtered]
    total    = sum(all_lens)
    n_take   = min(n_samples, total)
    flat_idx = np.sort(rng.choice(total, n_take, replace=False))
    D        = filtered[0].shape[1]
    result   = np.empty((n_take, D), dtype=np.float32)
    cumsum = out = 0
    it = iter(flat_idx); nfi = next(it, None)
    for pi, L in enumerate(all_lens):
        pe = cumsum + L
        while nfi is not None and nfi < pe:
            result[out] = filtered[pi][nfi - cumsum]; out += 1; nfi = next(it, None)
        cumsum = pe
        if nfi is None: break
    print(f"  Sampled {out:,} / {total:,} patches "
          f"({out/max(total,1)*100:.1f}% of norm-filtered pool)")
    return torch.from_numpy(result[:out])

train_raw  = sample_patches(fused_train, RVQ_TRAINING_SAMPLES)
train_data = F.normalize(train_raw, dim=-1)
del train_raw, fused_train; gc.collect()
print(f"  train_data: {train_data.shape}  (L2-normalized, unit hypersphere)")

# ==============================================================================
# SHARED UTILITIES: RANKING LOSSES (used by both KMeans and SPN pipelines)
# ==============================================================================

# ──────────────────────────────────────────────────────────────────────────────
# WHY RANKING LOSSES FOR RVQ CODEBOOK TRAINING
# ──────────────────────────────────────────────────────────────────────────────
#
# The fundamental mismatch in prior RVQ work:
#
#   Training objective:  min MSE(z_q, x)          [compression]
#   Inference objective: max MaxSim(q, z_q)        [ranking]
#
# These conflict because:
#   1. MSE-optimal quantization minimizes ||x - z_q||² uniformly across the
#      embedding space — it does NOT distinguish semantically discriminative
#      directions from irrelevant ones for retrieval.
#   2. MaxSim(q, d) = Σ_patch max_j cos(q_j, d_patch_k) depends on the
#      ANGULAR structure of codebook vectors on the sphere. Two codebook
#      vectors that are "close" in L2 may collapse MaxSim discrimination even
#      if individual MSE is low.
#   3. Hard negatives (visually similar but semantically different pages) have
#      similar embeddings → MSE loss ignores this; ranking loss directly
#      penalizes score collapse.
#
# Solution: augment the compression loss with two ranking-aware losses:
#
# L_rank (Contrastive/Triplet):
#   L_rank = E[max(0, γ - MaxSim(q,pos) + MaxSim(q,neg))]
#   - γ = RANK_MARGIN (margin)
#   - Uses in-batch negatives: for anchor i, positives = same document group,
#     negatives = other documents in batch
#   - In the RVQ context: q = raw query patch embedding,
#     pos/neg = quantized document embeddings from same/different docs
#   - Gradient flows through codebook vectors via quantized reconstruction
#
# L_OT (Wasserstein / Optimal Transport):
#   Motivation: MSE treats all reconstruction errors equally in Euclidean space.
#   But for ranking, we care about DISTRIBUTIONAL alignment between the
#   distribution of quantized embeddings and original embeddings.
#   Wasserstein distance is the natural metric between distributions and is
#   provably better than MSE at preserving the geometric structure needed
#   for MaxSim ranking (Villani 2008; Frogner et al. 2015).
#
#   Implementation: Sinkhorn algorithm (entropic regularized OT, Cuturi 2013)
#   Cost matrix: C[i,j] = 1 - cos(z_q_i, x_j)  (angular distance on sphere)
#   OT plan: π* = argmin_{π ∈ U(a,b)} Σ_{i,j} C[i,j] π[i,j] + ε H(π)
#   Loss: L_OT = Σ_{i,j} C[i,j] π*[i,j]
#
#   Novelty: applying Sinkhorn-OT to RVQ codebook training for document retrieval
#   is not found in prior literature. Closest work:
#   - SWAV uses optimal transport for prototype assignment (different problem)
#   - Audio codec papers use MSE + commitment exclusively
#   - POT (Python OT library) applied to NLP matching, not codebook training
#
# Total loss:
#   L_total = L_recon + λ_rank * L_rank + λ_OT * L_OT + 0.25 * L_commit
#
# where L_recon = MSE (kept for stability / gradient signal magnitude)
# ──────────────────────────────────────────────────────────────────────────────

def maxsim_score(query_patches: torch.Tensor,
                 doc_patches:   torch.Tensor) -> torch.Tensor:
    """
    Compute ColPali-style MaxSim score.

    query_patches : (Q, D)  — query patch embeddings (L2-normalized)
    doc_patches   : (P, D)  — document patch embeddings (quantized or raw)

    MaxSim(q, d) = Σ_{j=1}^{Q}  max_{k=1}^{P}  cos(q_j, d_k)
                = Σ_j  max_k  (q_j · d_k)   [since both normalized]

    Returns: scalar score
    """
    # (Q, P) cosine similarity matrix
    sim = torch.mm(query_patches, doc_patches.t())
    # Max over document patches, sum over query patches
    return sim.max(dim=1).values.sum()


def batch_maxsim(queries: torch.Tensor,
                 docs:    torch.Tensor,
                 n_q_patches: int) -> torch.Tensor:
    """
    Vectorized MaxSim over a batch.

    queries : (B * n_q_patches, D)  — flattened query patches
    docs    : (B * n_d_patches, D)  — flattened doc patches
    n_q_patches: patches per query

    Returns: (B,) MaxSim scores
    """
    B = queries.shape[0] // n_q_patches
    n_d_patches = docs.shape[0] // B

    q = queries.view(B, n_q_patches, -1)   # (B, Q, D)
    d = docs.view(B, n_d_patches, -1)       # (B, P, D)

    # (B, Q, P) cosine similarity
    sim = torch.bmm(q, d.transpose(1, 2))   # (B, Q, P)
    # Max over P (doc patches), sum over Q (query patches)
    return sim.max(dim=2).values.sum(dim=1)  # (B,)


def ranking_contrastive_loss(x_orig: torch.Tensor,
                              z_q:    torch.Tensor,
                              margin: float = RANK_MARGIN,
                              n_neg:  int   = RANK_NEG_SAMPLES) -> torch.Tensor:
    """
    In-batch triplet ranking loss for MaxSim preservation.

    Intuition:
      x_orig  = raw (unquantized) embeddings → serve as "queries"
      z_q     = quantized embeddings → serve as "documents"
      For each anchor i:
        positive = z_q[i]       (quantized version of same embedding)
        negatives = z_q[j≠i]   (quantized embeddings from other docs)
      Loss = max(0, margin - MaxSim(x_i, z_q[i]) + MaxSim(x_i, z_q[j]))

    In the RVQ context this directly penalizes cases where the quantized
    representation of doc i looks MORE similar to query i than quantized
    doc j does — i.e., it preserves the ranking induced by the original
    embedding space after quantization.

    x_orig : (N, D) raw patch embeddings (normalized)
    z_q    : (N, D) quantized patch embeddings (normalized)

    Note: here each row is treated as a single-patch "document" for efficiency.
    For multi-patch MaxSim, use batch_maxsim above at eval time.
    """
    N = x_orig.shape[0]

    # Anchor: x_orig[i], positive: z_q[i], negatives: z_q[rand j≠i]
    # Cosine similarity (both normalized → dot product)
    pos_sim = (x_orig * z_q).sum(dim=-1)          # (N,) anchor-positive scores

    # Sample n_neg negatives per anchor (in-batch)
    neg_idx = torch.zeros(N, n_neg, dtype=torch.long, device=x_orig.device)
    for i in range(N):
        pool = torch.arange(N, device=x_orig.device)
        pool = pool[pool != i]
        perm = torch.randperm(len(pool), device=x_orig.device)[:n_neg]
        neg_idx[i] = pool[perm]

    # (N, n_neg, D) negative quantized embeddings
    neg_z = z_q[neg_idx.view(-1)].view(N, n_neg, -1)

    # (N, n_neg) anchor-negative scores
    neg_sim = (x_orig.unsqueeze(1) * neg_z).sum(dim=-1)

    # Triplet hinge: max(0, margin - pos + neg)
    # Use hardest negative (highest neg_sim) for max-margin training
    hardest_neg_sim = neg_sim.max(dim=1).values   # (N,)
    loss = F.relu(margin - pos_sim + hardest_neg_sim).mean()
    return loss


def sinkhorn_log(log_alpha: torch.Tensor,
                 n_iters:   int   = OT_SINKHORN_ITERS,
                 eps:       float = OT_SINKHORN_EPS) -> torch.Tensor:
    """
    Log-domain Sinkhorn algorithm for numerical stability.

    Solves entropic regularized OT:
      min_{π} Σ_{ij} C_{ij} π_{ij}  + ε * KL(π || a⊗b)
    subject to π 1 = a, π^T 1 = b  (marginal constraints)

    Here: a = b = uniform = 1/N  (equal weights on both sides)

    log_alpha : (N, M) log cost matrix = log(C) where C[i,j] is transport cost
    Returns  : (N, M) log OT plan
    """
    N, M = log_alpha.shape
    log_a = torch.full((N,), -np.log(N), device=log_alpha.device, dtype=log_alpha.dtype)
    log_b = torch.full((M,), -np.log(M), device=log_alpha.device, dtype=log_alpha.dtype)

    log_u = torch.zeros(N, device=log_alpha.device, dtype=log_alpha.dtype)
    log_v = torch.zeros(M, device=log_alpha.device, dtype=log_alpha.dtype)

    for _ in range(n_iters):
        # u update: log_u = log_a - logsumexp(log_alpha + log_v, dim=1)
        log_u = log_a - torch.logsumexp(
            log_alpha + log_v.unsqueeze(0), dim=1)
        # v update: log_v = log_b - logsumexp(log_alpha + log_u, dim=0)
        log_v = log_b - torch.logsumexp(
            log_alpha + log_u.unsqueeze(1), dim=0)

    # log transport plan: log_T[i,j] = log_alpha[i,j] + log_u[i] + log_v[j]
    log_T = log_alpha + log_u.unsqueeze(1) + log_v.unsqueeze(0)
    return log_T


def wasserstein_ot_loss(x_orig: torch.Tensor,
                         z_q:    torch.Tensor,
                         eps:    float = OT_SINKHORN_EPS,
                         n_iters: int  = OT_SINKHORN_ITERS) -> torch.Tensor:
    """
    Sinkhorn-Wasserstein loss between distributions of original and quantized
    embeddings. Operates on a random subsample for efficiency.

    Theory:
      W_ε(μ, ν) = min_{π ∈ Π(μ,ν)} E_{(x,y)~π}[C(x,y)] + ε * KL(π || μ⊗ν)
    where:
      μ = empirical distribution of x_orig  (original embeddings)
      ν = empirical distribution of z_q     (quantized embeddings)
      C(x, y) = 1 - cos(x, y)              (angular cost on sphere, ∈ [0,2])

    This measures how much "work" is needed to transform the distribution of
    quantized embeddings into the distribution of original embeddings.
    Lower W_ε → quantization preserves the geometric structure of the
    embedding space → better MaxSim ranking after quantization.

    Gradient flows through z_q via the cost matrix C, encouraging codebook
    vectors to be positioned such that OT plan is low-cost.

    Efficiency: subsample to OT_BATCH_SIZE²  (O(N²) cost matrix is the
    bottleneck; 512×512 is fast on GPU, 2048×2048 needs mixed precision).

    x_orig : (N, D)
    z_q    : (N, D) — must have grad
    """
    N = x_orig.shape[0]

    # Subsample for efficiency (OT cost is O(N^2 * D))
    ot_n = min(N, 512)
    idx  = torch.randperm(N, device=x_orig.device)[:ot_n]
    x    = x_orig[idx]  # (ot_n, D), no grad needed
    z    = z_q[idx]     # (ot_n, D), grad flows here

    # Angular cost matrix: C[i,j] = 1 - cos(x_i, z_j) ∈ [0, 2]
    # Lower cost when quantized embedding z_j is close in direction to x_i
    cos_sim = torch.mm(x, z.t())         # (ot_n, ot_n)
    C = 1.0 - cos_sim                    # angular distance ∈ [0, 2]

    # Log-cost for Sinkhorn: log_alpha = -C / eps
    # (entropic regularization makes the problem smooth and tractable)
    log_alpha = -C / eps                 # (ot_n, ot_n)

    # Sinkhorn iterations in log-domain (numerically stable)
    with torch.no_grad():
        log_T = sinkhorn_log(log_alpha.detach(), n_iters=n_iters, eps=eps)

    # Transport plan (normalized)
    T = log_T.exp()                      # (ot_n, ot_n), no grad

    # OT loss = Σ_{i,j} C[i,j] * T[i,j]
    # Gradient through C → through z (via cos_sim)
    loss = (C * T).sum()
    return loss


# ==============================================================================
# STEP 5A: KMEANS-BASED RVQ TRAINING (BASELINE)
# ==============================================================================
print("\n" + "="*70)
print("STEP 5A: KMeans-initialized RVQ (BASELINE)")
print("="*70)

# ──────────────────────────────────────────────────────────────────────────────
# KMeans Baseline Design
# ──────────────────────────────────────────────────────────────────────────────
#
# Uses the built-in kmeans_init=True of vector_quantize_pytorch.
# KMeans centroids are computed once before training and used to seed codebooks.
# Then RVQ is fine-tuned with the combined loss:
#   L = L_recon + λ_rank * L_rank + λ_OT * L_OT + 0.25 * L_commit
#
# Known limitations of KMeans init (included here for fair comparison):
#   1. Centroids not constrained to unit sphere after averaging
#   2. High-density bias → rare document types underrepresented
#   3. No entropy regularization → potential dead codes
#
# Despite these limitations, KMeans is the strong baseline used in
# SoundStream (2022), EnCodec (2022) and most downstream RVQ work.
# Our contribution: even with KMeans init, adding ranking losses improves
# retrieval quality — demonstrating that the loss function matters
# independently of the initialization method.
# ──────────────────────────────────────────────────────────────────────────────

def rvq_forward_with_dropout_and_ranking(
    model:        ResidualVQ,
    x:            torch.Tensor,
    n_drop_suffix: int,
    use_rank_loss: bool = True,
    use_ot_loss:   bool = True,
) -> tuple:
    """
    Forward pass: compression + ranking losses.

    x             : (N, D) L2-normalized patch embeddings
    n_drop_suffix : number of tail quantizers dropped (0 = full)
    use_rank_loss : whether to compute contrastive ranking loss
    use_ot_loss   : whether to compute Sinkhorn-OT loss

    Returns: (total_loss, dict of component losses, n_active_quantizers)

    Loss composition:
      L_recon  = MSE(z_q, x)               → geometric fidelity
      L_align  = 1 - cos(z_q, x)           → direction alignment (MaxSim proxy)
      L_commit = commitment loss from VQ    → codebook stability
      L_rank   = triplet margin ranking     → MaxSim ordering preservation
      L_OT     = Sinkhorn-Wasserstein       → distributional alignment
    """
    NQ       = len(model.layers)
    n_active = max(1, NQ - n_drop_suffix)

    residual   = x.clone()
    z_q_cumsum = torch.zeros_like(x)
    all_commits = []

    for k in range(n_active):
        z_q_k, _, commit_k = model.layers[k](residual.unsqueeze(1))
        z_q_k      = z_q_k.squeeze(1)
        z_q_cumsum = z_q_cumsum + z_q_k
        residual   = residual - z_q_k
        if commit_k is not None:
            all_commits.append(commit_k.sum())

    # ── Compression losses (MSE + cosine alignment) ───────────────────────
    l_recon  = F.mse_loss(z_q_cumsum.float(), x.float().detach())
    l_align  = (1.0 - F.cosine_similarity(
                    z_q_cumsum.float(), x.float(), dim=-1)).mean()
    l_commit = (torch.stack(all_commits).mean()
                if all_commits else torch.tensor(0., device=x.device))

    # ── Ranking loss: contrastive triplet on MaxSim ───────────────────────
    # Normalize z_q before ranking losses (required for cosine similarity)
    z_q_norm = F.normalize(z_q_cumsum.float(), dim=-1)
    x_norm   = F.normalize(x.float(), dim=-1)

    if use_rank_loss and RANK_LOSS_WEIGHT > 0:
        l_rank = ranking_contrastive_loss(
            x_norm, z_q_norm,
            margin=RANK_MARGIN,
            n_neg=RANK_NEG_SAMPLES,
        )
    else:
        l_rank = torch.tensor(0., device=x.device)

    # ── OT loss: Sinkhorn-Wasserstein distributional alignment ───────────
    if use_ot_loss and OT_LOSS_WEIGHT > 0:
        l_ot = wasserstein_ot_loss(
            x_norm.detach(), z_q_norm,
            eps=OT_SINKHORN_EPS,
            n_iters=OT_SINKHORN_ITERS,
        )
    else:
        l_ot = torch.tensor(0., device=x.device)

    total = (l_recon
             + 0.5  * l_align
             + 0.25 * l_commit
             + RANK_LOSS_WEIGHT * l_rank
             + OT_LOSS_WEIGHT   * l_ot)

    losses = {
        'recon':  l_recon.item(),
        'align':  l_align.item(),
        'commit': l_commit.item(),
        'rank':   l_rank.item(),
        'ot':     l_ot.item(),
    }
    return total, losses, n_active


def train_rvq_kmeans(NQ: int, train_data: torch.Tensor,
                     dev: str, emb_dim: int) -> ResidualVQ:
    """
    Train RVQ with KMeans init + quantizer dropout + ranking losses.

    kmeans_init=True: built-in KMeans seeds codebooks before training starts.
    This is the standard approach from SoundStream/EnCodec literature.
    We augment with ranking losses for retrieval-aware training.
    """
    model = ResidualVQ(
        dim                     = emb_dim,
        num_quantizers          = NQ,
        codebook_size           = RVQ_CODEBOOK_SIZE,
        kmeans_init             = True,      # ← Standard KMeans init
        kmeans_iters            = 10,        # ← 10 iterations of Lloyd's
        threshold_ema_dead_code = 2,
        commitment_weight       = 0.5,
        learnable_codebook      = True,
        ema_update              = False,     # gradient-based update
    ).to(dev)

    print(f"  KMeans init: 10 iterations of Lloyd's algorithm")
    print(f"  + Quantizer dropout training")
    print(f"  + Contrastive ranking loss (λ={RANK_LOSS_WEIGHT}, margin={RANK_MARGIN})")
    print(f"  + Sinkhorn-OT loss (λ={OT_LOSS_WEIGHT}, ε={OT_SINKHORN_EPS})")

    n     = train_data.shape[0]
    n_val = max(RVQ_TRAINING_BATCH_SIZE, int(n * 0.1))
    perm  = torch.randperm(n)
    t_data = train_data[perm[n_val:]]
    v_data = train_data[perm[:n_val]]

    from torch.utils.data import DataLoader, TensorDataset
    import torch.amp as amp

    loader = DataLoader(
        TensorDataset(t_data),
        batch_size = RVQ_TRAINING_BATCH_SIZE,
        shuffle    = True, drop_last=True,
        pin_memory = (dev == 'cuda'))

    opt   = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-5)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(
        opt, T_max=RVQ_TRAINING_EPOCHS, eta_min=1e-5)
    scaler = amp.GradScaler(device=dev, enabled=(dev == 'cuda'))

    best_val   = float('inf')
    patience   = 0
    best_state = None
    rng_drop   = random.Random(42)

    for epoch in range(RVQ_TRAINING_EPOCHS):
        model.train()

        if epoch < RVQ_DROPOUT_WARMUP_EPS:
            drop_p = 0.0
        else:
            ramp   = min(1.0, (epoch - RVQ_DROPOUT_WARMUP_EPS) / 20.0)
            drop_p = RVQ_DROPOUT_P * ramp

        ep_losses = {k: 0. for k in ['recon', 'align', 'commit', 'rank', 'ot']}
        ep_total  = 0.
        nb = 0

        for (batch,) in loader:
            batch = batch.to(dev, non_blocking=True)

            n_drop = 0
            if drop_p > 0 and rng_drop.random() < drop_p:
                n_drop = min(
                    NQ - 1,
                    int(np.random.geometric(p=1 - drop_p * 0.5)) - 1)

            opt.zero_grad(set_to_none=True)
            with amp.autocast(device_type=dev, enabled=(dev == 'cuda')):
                loss, comp_losses, n_act = rvq_forward_with_dropout_and_ranking(
                    model, batch, n_drop,
                    use_rank_loss=True, use_ot_loss=True)

            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update()

            ep_total += loss.item()
            for k, v in comp_losses.items():
                ep_losses[k] += v
            nb += 1

        sched.step()

        model.eval()
        with torch.no_grad():
            vi = torch.randperm(v_data.shape[0])[:min(4096, v_data.shape[0])]
            vb = v_data[vi].to(dev)
            with amp.autocast(device_type=dev, enabled=(dev == 'cuda')):
                val_loss, val_comp, _ = rvq_forward_with_dropout_and_ranking(
                    model, vb, n_drop_suffix=0,
                    use_rank_loss=False, use_ot_loss=False)  # val: recon only
            val_loss = val_loss.item()

        if epoch == 0 or (epoch + 1) % 10 == 0:
            lr_now = sched.get_last_lr()[0]
            el = {k: v/max(nb,1) for k, v in ep_losses.items()}
            print(f"  [KMeans NQ={NQ}] ep{epoch+1:3d}  "
                  f"total={ep_total/max(nb,1):.5f}  "
                  f"recon={el['recon']:.5f}  "
                  f"rank={el['rank']:.5f}  "
                  f"ot={el['ot']:.5f}  "
                  f"val={val_loss:.5f}  "
                  f"drop_p={drop_p:.2f}  lr={lr_now:.2e}")

        if val_loss < best_val - 1e-6:
            best_val   = val_loss
            patience   = 0
            best_state = {k: v.cpu().clone()
                          for k, v in model.state_dict().items()}
        else:
            patience += 1

        if patience >= RVQ_EARLY_STOP_PATIENCE:
            print(f"  [KMeans NQ={NQ}] early-stop ep{epoch+1}  best_val={best_val:.5f}")
            break

    if best_state:
        model.load_state_dict({k: v.to(dev) for k, v in best_state.items()})

    model.eval()
    return model


# ==============================================================================
# STEP 5B: SPHERICAL PROTOTYPE NETWORK (SPN) INITIALIZATION
# ==============================================================================
print("\n" + "="*70)
print("STEP 5B: Spherical Prototype Network (SPN) codebook initialization")
print("="*70)

# ──────────────────────────────────────────────────────────────────────────────
# SPN Architecture and Rationale
# ──────────────────────────────────────────────────────────────────────────────
#
# Core insight: KMeans on unit hypersphere is suboptimal because:
#   1. GEOMETRIC MISMATCH: KMeans minimizes ||x - μ||² (Euclidean, flat space)
#      but data lives on S^{D-1} (unit sphere, curved). After re-normalization,
#      convergence guarantees of Lloyd's algorithm are broken.
#   2. DENSITY BIAS: centroids concentrate in high-density regions; rare
#      document types (diagrams, formulas) are underrepresented → dead codes.
#   3. HARD ASSIGNMENT: no gradient through assignment → codebook entries
#      don't capture structure between cluster boundaries.
#
# SPN replaces KMeans with:
#   - Soft assignment: q_i = softmax((z_i · P^T) / τ)
#   - Entropy maximization: L_H = -H(mean_assignment) → uniform codebook usage
#   - Spherical projection: P ← P / ||P|| after each gradient step
#   - Temperature annealing: τ: 1.0 → 0.1 (exploration → commitment)
#   - Per-residual-stage training: stage k SPN sees residual_k distribution
#
# Novel contributions vs. prior work:
#   - SwAV (Caron 2020): prototype SSL, cross-view prediction, no RVQ
#   - DINO (Caron 2021): student-teacher, no codebook init application
#   - Per-stage residual SPN: not found in any prior RVQ/codec paper
#   - L_spread (spherical diversity): prevent prototype collapse on sphere
# ──────────────────────────────────────────────────────────────────────────────

class SphericalPrototypeNetwork(nn.Module):
    """
    Learn prototype distribution on unit hypersphere.
    Replaces KMeans for RVQ codebook initialization.

    Parameters
    ----------
    emb_dim    : input embedding dimension (L2-normalized)
    n_proto    : number of prototypes = codebook_size for one RVQ stage
    hidden_dim : MLP hidden dimension
    """
    def __init__(self, emb_dim: int, n_proto: int, hidden_dim: int = 512):
        super().__init__()
        self.n_proto  = n_proto
        self.emb_dim  = emb_dim

        # Encoder MLP with skip connection
        # Purpose: learn which dimensions are discriminative per prototype
        # Skip keeps sphere structure intact (skip_weight ≈ 0 initially)
        self.encoder = nn.Sequential(
            nn.LayerNorm(emb_dim),
            nn.Linear(emb_dim, hidden_dim),
            nn.GELU(),
            nn.LayerNorm(hidden_dim),
            nn.Linear(hidden_dim, emb_dim),
        )
        self.skip_weight = nn.Parameter(torch.tensor(0.1))

        # Prototypes: (n_proto, emb_dim), sphere-projected after each step
        proto_init = torch.randn(n_proto, emb_dim)
        proto_init = F.normalize(proto_init, dim=-1)
        self.prototypes = nn.Parameter(proto_init)

    @property
    def spherical_prototypes(self) -> torch.Tensor:
        return F.normalize(self.prototypes, dim=-1)

    def encode(self, x: torch.Tensor) -> torch.Tensor:
        h = self.encoder(x)
        return F.normalize(x + self.skip_weight.abs() * h, dim=-1)

    def forward(self, x: torch.Tensor, temperature: float = 1.0):
        z = self.encode(x)                        # (N, D)
        P = self.spherical_prototypes             # (K, D)

        # Cosine similarity / temperature → soft assignment
        sim = torch.mm(z, P.t()) / temperature   # (N, K)
        q   = F.softmax(sim, dim=-1)              # (N, K) per-sample distribution
        q_bar = q.mean(dim=0)                     # (K,) mean assignment per prototype

        # L_entropy: maximize entropy over prototypes → uniform codebook usage
        # H = -Σ_k q̄_k log(q̄_k), maximized when q̄ = uniform = 1/K
        eps = 1e-8
        H   = -(q_bar * (q_bar + eps).log()).sum()
        l_entropy = -H   # minimize -H = maximize H

        # L_recon: prototypes should represent their assigned patches
        # Straight-through estimator: gradient flows through P, not argmax
        idx    = q.argmax(dim=-1)
        p_hard = P[idx]
        l_recon = F.mse_loss(p_hard, z.detach())

        # L_spread: prevent prototype collapse on sphere
        # Penalize prototype pairs with cosine similarity > 0 (too close)
        sim_pp_grad = torch.mm(P, P.t())
        sim_pp_grad.fill_diagonal_(-1.0)
        l_spread = F.relu(sim_pp_grad).mean()

        loss = l_entropy + SPN_ENTROPY_WEIGHT * l_recon + 0.1 * l_spread

        return loss, q, idx, {
            'l_entropy':    l_entropy.item(),
            'l_recon':      l_recon.item(),
            'l_spread':     l_spread.item(),
            'entropy':      H.item(),
            'max_entropy':  float(np.log(self.n_proto)),
        }

    @torch.no_grad()
    def seed_from_data(self, data: torch.Tensor):
        """Reinitialize prototypes from random data samples (on-manifold init)."""
        n_data = data.shape[0]
        idx    = torch.randperm(n_data)[:self.n_proto]
        seeds  = F.normalize(data[idx].float(), dim=-1)
        self.prototypes.data.copy_(seeds)
        print(f"    SPN seeded from {self.n_proto} random data points (on-sphere)")


def train_spn_for_stage(
    residual_data: torch.Tensor,
    n_proto:       int,
    emb_dim:       int,
    dev:           str,
    stage_idx:     int,
) -> torch.Tensor:
    """
    Train SPN for one RVQ stage on its specific residual distribution.
    Returns: (n_proto, emb_dim) spherical prototypes → codebook initialization.

    Key novelty: each stage receives its own residual:
      stage 0: residual = x (original embeddings)
      stage k: residual = x - Σ_{j<k} z_q_j  (decreasing magnitude)
    → prototypes of stage k are tuned for the distribution of residual_k,
      not for raw embeddings (as in all prior RVQ work).
    """
    import torch.amp as amp
    from torch.utils.data import DataLoader, TensorDataset

    spn = SphericalPrototypeNetwork(
        emb_dim=emb_dim, n_proto=n_proto, hidden_dim=SPN_HIDDEN_DIM
    ).to(dev)
    spn.seed_from_data(residual_data.to(dev))

    loader = DataLoader(
        TensorDataset(residual_data),
        batch_size=SPN_BATCH_SIZE, shuffle=True, drop_last=True,
        pin_memory=(dev == 'cuda'),
    )

    opt   = torch.optim.AdamW(spn.parameters(), lr=SPN_LR, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(
        opt, T_max=SPN_EPOCHS, eta_min=SPN_LR / 10)
    scaler = amp.GradScaler(device=dev, enabled=(dev == 'cuda'))

    best_entropy = -float('inf')
    best_proto   = None

    for epoch in range(SPN_EPOCHS):
        spn.train()
        frac = epoch / max(SPN_EPOCHS - 1, 1)
        tau  = SPN_TEMP_START + frac * (SPN_TEMP_END - SPN_TEMP_START)
        ep_loss = ep_H = nb = 0

        for (batch,) in loader:
            batch = batch.to(dev, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with amp.autocast(device_type=dev, enabled=(dev == 'cuda')):
                loss, _, _, info = spn(batch, temperature=tau)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(spn.parameters(), 1.0)
            with torch.no_grad():
                spn.prototypes.data = F.normalize(spn.prototypes.data, dim=-1)
            scaler.step(opt); scaler.update()
            ep_loss += loss.item()
            ep_H    += info['entropy']
            nb      += 1

        sched.step()
        mean_H = ep_H / max(nb, 1)
        if mean_H > best_entropy:
            best_entropy = mean_H
            best_proto   = spn.spherical_prototypes.detach().cpu().clone()

        if epoch == 0 or (epoch + 1) % 5 == 0:
            max_H = info['max_entropy']
            pct_H = mean_H / max_H * 100 if max_H > 0 else 0
            print(f"    [SPN stage={stage_idx}] ep{epoch+1:3d}  "
                  f"loss={ep_loss/max(nb,1):.4f}  "
                  f"H={mean_H:.3f}/{max_H:.3f} ({pct_H:.0f}%)  τ={tau:.3f}")

    with torch.no_grad():
        spn.eval()
        val_batch = residual_data[:min(4096, len(residual_data))].to(dev)
        _, q_final, idx_final, _ = spn(val_batch, temperature=SPN_TEMP_END)
        used = idx_final.unique().numel()
        dead = n_proto - used
        h_pct = best_entropy / info['max_entropy'] * 100
    print(f"    [SPN stage={stage_idx}] done  "
          f"best_H={best_entropy:.3f} ({h_pct:.0f}%)  "
          f"dead_codes={dead}/{n_proto} "
          f"({'⚠️' if dead > n_proto * 0.1 else '✅'})")

    del spn, loader; gc.collect(); torch.cuda.empty_cache()
    return best_proto  # (n_proto, emb_dim), CPU


def initialize_rvq_codebooks_with_spn(
    rvq_model: ResidualVQ,
    train_data: torch.Tensor,
    NQ: int, emb_dim: int, dev: str,
) -> ResidualVQ:
    """
    Replace all KMeans init with per-stage SPN init.

    Per-stage algorithm:
      residual_0 = train_data
      for k = 0..NQ-1:
        1. Train SPN_k on normalized(residual_k) → prototypes P_k
        2. Inject P_k into rvq_model.layers[k].codebook
        3. Compute hard assignment via cosine similarity
        4. Update residual_{k+1} = residual_k - P_k[argmax_cosine]
    """
    print(f"\n  Per-stage SPN initialization for NQ={NQ} stages...")
    residual = train_data.clone()

    spn_n = min(len(residual), SPN_BATCH_SIZE * SPN_EPOCHS)
    perm  = torch.randperm(len(residual))[:spn_n]
    spn_residual = residual[perm]

    for k in range(NQ):
        print(f"\n  {'─'*50}")
        print(f"  Stage {k}/{NQ-1}: SPN on residual_k distribution")

        residual_norm = F.normalize(spn_residual, dim=-1)
        proto_k = train_spn_for_stage(
            residual_data=residual_norm,
            n_proto=RVQ_CODEBOOK_SIZE,
            emb_dim=emb_dim, dev=dev, stage_idx=k,
        )

        # Inject prototypes into RVQ codebook layer k
        vq_layer    = rvq_model.layers[k]
        proto_k_gpu = proto_k.to(dev)
        injected    = False

        for attr_path in ['_codebook.embed', '_codebook.embed_avg', 'codebook']:
            try:
                obj = vq_layer
                parts = attr_path.split('.')
                for part in parts[:-1]: obj = getattr(obj, part)
                param = getattr(obj, parts[-1])
                target = proto_k_gpu if param.data.shape == proto_k_gpu.shape \
                    else proto_k_gpu.t() if param.data.shape == proto_k_gpu.t().shape \
                    else None
                if target is not None:
                    if isinstance(param, nn.Parameter): param.data.copy_(target)
                    else: param.copy_(target)
                    injected = True
                    print(f"    Injected via {attr_path}")
                    break
            except (AttributeError, RuntimeError):
                continue

        if not injected:
            for n, p in vq_layer.named_parameters():
                if RVQ_CODEBOOK_SIZE in p.shape and p.data.dim() >= 2:
                    flat = p.data
                    while flat.dim() > 2: flat = flat[0]
                    if flat.shape[0] == RVQ_CODEBOOK_SIZE:
                        flat.copy_(proto_k_gpu); injected = True
                    elif flat.dim() == 2 and flat.shape[1] == RVQ_CODEBOOK_SIZE:
                        flat.copy_(proto_k_gpu.t()); injected = True
                    if injected:
                        print(f"    Injected via named_param {n} (fallback)")
                        break

        if not injected:
            print(f"    [WARN] Could not inject stage {k} — using random init")

        # Simulate stage k quantization → compute residual_{k+1}
        with torch.no_grad():
            proto_gpu_norm = F.normalize(proto_k_gpu, dim=-1)
            chunk_size = 8192
            z_q_k_list = []
            for start in range(0, len(spn_residual), chunk_size):
                chunk = F.normalize(spn_residual[start:start+chunk_size].to(dev), dim=-1)
                sims  = torch.mm(chunk, proto_gpu_norm.t())
                idx   = sims.argmax(dim=-1)
                z_q_k_list.append(proto_gpu_norm[idx].cpu())
            z_q_k = torch.cat(z_q_k_list, dim=0)
            spn_residual = spn_residual - z_q_k

        del proto_k_gpu, z_q_k; gc.collect(); torch.cuda.empty_cache()

    print(f"\n  SPN initialization complete for all {NQ} stages")
    return rvq_model


# ==============================================================================
# STEP 5C: SPN-INITIALIZED RVQ TRAINING WITH RANKING LOSSES
# ==============================================================================

def train_rvq_spn(NQ: int, train_data: torch.Tensor,
                  dev: str, emb_dim: int) -> ResidualVQ:
    """
    Train RVQ with:
      Phase 1: Per-stage SPN initialization (replaces KMeans)
      Phase 2: Dropout training + Contrastive ranking loss + Sinkhorn-OT loss

    Total loss per batch:
      L = L_recon + 0.5*L_align + 0.25*L_commit
        + λ_rank * L_rank    (contrastive triplet, MaxSim-aware)
        + λ_OT   * L_OT      (Sinkhorn-Wasserstein, distributional)
    """
    model = ResidualVQ(
        dim                     = emb_dim,
        num_quantizers          = NQ,
        codebook_size           = RVQ_CODEBOOK_SIZE,
        kmeans_init             = False,   # SPN replaces KMeans
        kmeans_iters            = 0,
        threshold_ema_dead_code = 2,
        commitment_weight       = 0.5,
        learnable_codebook      = True,
        ema_update              = False,
    ).to(dev)

    # ── Phase 1: SPN initialization ───────────────────────────────────────
    print(f"\n  [SPN] Phase 1: Per-stage SPN initialization (NQ={NQ})")
    t_spn = time.perf_counter()
    model = initialize_rvq_codebooks_with_spn(
        model, train_data, NQ, emb_dim, dev)
    print(f"  SPN init done in {time.perf_counter()-t_spn:.1f}s")

    # ── Phase 2: RVQ training with dropout + ranking losses ───────────────
    print(f"\n  [SPN] Phase 2: RVQ training with dropout + ranking losses (NQ={NQ})")
    print(f"  + Contrastive ranking loss (λ={RANK_LOSS_WEIGHT}, margin={RANK_MARGIN})")
    print(f"  + Sinkhorn-OT loss (λ={OT_LOSS_WEIGHT}, ε={OT_SINKHORN_EPS})")

    n     = train_data.shape[0]
    n_val = max(RVQ_TRAINING_BATCH_SIZE, int(n * 0.1))
    perm  = torch.randperm(n)
    t_data = train_data[perm[n_val:]]
    v_data = train_data[perm[:n_val]]

    from torch.utils.data import DataLoader, TensorDataset
    import torch.amp as amp

    loader = DataLoader(
        TensorDataset(t_data),
        batch_size=RVQ_TRAINING_BATCH_SIZE, shuffle=True, drop_last=True,
        pin_memory=(dev == 'cuda'))

    opt   = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-5)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(
        opt, T_max=RVQ_TRAINING_EPOCHS, eta_min=1e-5)
    scaler = amp.GradScaler(device=dev, enabled=(dev == 'cuda'))

    best_val   = float('inf')
    patience   = 0
    best_state = None
    rng_drop   = random.Random(42)

    for epoch in range(RVQ_TRAINING_EPOCHS):
        model.train()

        if epoch < RVQ_DROPOUT_WARMUP_EPS:
            drop_p = 0.0
        else:
            ramp   = min(1.0, (epoch - RVQ_DROPOUT_WARMUP_EPS) / 20.0)
            drop_p = RVQ_DROPOUT_P * ramp

        ep_losses = {k: 0. for k in ['recon', 'align', 'commit', 'rank', 'ot']}
        ep_total  = 0.
        nb = 0

        for (batch,) in loader:
            batch = batch.to(dev, non_blocking=True)

            n_drop = 0
            if drop_p > 0 and rng_drop.random() < drop_p:
                n_drop = min(
                    NQ - 1,
                    int(np.random.geometric(p=1 - drop_p * 0.5)) - 1)

            opt.zero_grad(set_to_none=True)
            with amp.autocast(device_type=dev, enabled=(dev == 'cuda')):
                loss, comp_losses, n_act = rvq_forward_with_dropout_and_ranking(
                    model, batch, n_drop,
                    use_rank_loss=True, use_ot_loss=True)

            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update()

            ep_total += loss.item()
            for k, v in comp_losses.items():
                ep_losses[k] += v
            nb += 1

        sched.step()

        model.eval()
        with torch.no_grad():
            vi = torch.randperm(v_data.shape[0])[:min(4096, v_data.shape[0])]
            vb = v_data[vi].to(dev)
            with amp.autocast(device_type=dev, enabled=(dev == 'cuda')):
                val_loss, _, _ = rvq_forward_with_dropout_and_ranking(
                    model, vb, n_drop_suffix=0,
                    use_rank_loss=False, use_ot_loss=False)
            val_loss = val_loss.item()

        if epoch == 0 or (epoch + 1) % 10 == 0:
            lr_now = sched.get_last_lr()[0]
            el = {k: v/max(nb,1) for k, v in ep_losses.items()}
            print(f"  [SPN NQ={NQ}] ep{epoch+1:3d}  "
                  f"total={ep_total/max(nb,1):.5f}  "
                  f"recon={el['recon']:.5f}  "
                  f"rank={el['rank']:.5f}  "
                  f"ot={el['ot']:.5f}  "
                  f"val={val_loss:.5f}  "
                  f"drop_p={drop_p:.2f}  lr={lr_now:.2e}")

        if val_loss < best_val - 1e-6:
            best_val   = val_loss
            patience   = 0
            best_state = {k: v.cpu().clone()
                          for k, v in model.state_dict().items()}
        else:
            patience += 1

        if patience >= RVQ_EARLY_STOP_PATIENCE:
            print(f"  [SPN NQ={NQ}] early-stop ep{epoch+1}  best_val={best_val:.5f}")
            break

    if best_state:
        model.load_state_dict({k: v.to(dev) for k, v in best_state.items()})

    model.eval()
    return model


# ==============================================================================
# STEP 6: CODEBOOK EXTRACTION + DIAGNOSTICS (shared)
# ==============================================================================

def extract_codebooks(model: ResidualVQ, NQ: int, dev: str) -> list:
    model.eval()
    cbs = []
    with torch.no_grad():
        for qi in range(NQ):
            vq = model.layers[qi]; cb = None
            for attr in ['_codebook.embed', '_codebook.embed_avg', 'codebook']:
                try:
                    obj = vq
                    for part in attr.split('.'): obj = getattr(obj, part)
                    obj = obj.detach().float()
                    if obj.dim() == 3: obj = obj[0]
                    if obj.dim() == 2 and obj.shape[0] == RVQ_CODEBOOK_SIZE:
                        cb = obj; break
                    if obj.dim() == 2 and obj.shape[1] == RVQ_CODEBOOK_SIZE:
                        cb = obj.t(); break
                except (AttributeError, IndexError):
                    continue
            if cb is None:
                for n, p in vq.named_parameters():
                    if RVQ_CODEBOOK_SIZE in p.shape:
                        obj = p.detach().float()
                        while obj.dim() > 2: obj = obj[0]
                        if obj.shape[0] == RVQ_CODEBOOK_SIZE: cb = obj; break
                        if obj.dim() == 2 and obj.shape[1] == RVQ_CODEBOOK_SIZE:
                            cb = obj.t(); break
            if cb is None:
                raise RuntimeError(f"Cannot extract codebook stage={qi}")
            cbs.append(cb.to(dev))
    return cbs


@torch.no_grad()
def codebook_diagnostics(codebooks: list, val_sample: torch.Tensor,
                          dev: str) -> dict:
    """
    Comprehensive codebook quality metrics.

    Metrics:
      perplexity_per_stage : 2^H(usage), target > 0.5 * CB_size
      stage_recon_mse      : per-stage residual MSE
      dead_codes_per_stage : codes never assigned in val set
      total_recon_mse      : cumulative reconstruction MSE
      cosine_alignment     : cos(z_q, x), proxy for MaxSim preservation
      maxsim_preservation  : ratio MaxSim(z_q) / MaxSim(x) [1.0 = perfect]
    """
    NQ = len(codebooks)
    val_sample = val_sample.to(dev)
    residual   = val_sample.clone()
    z_q_sum    = torch.zeros_like(val_sample)
    perplexities = []
    stage_mses   = []
    dead_codes   = []

    for k, cb in enumerate(codebooks):
        inner  = torch.mm(residual, cb.t())
        res_sq = (residual ** 2).sum(-1, keepdim=True)
        cb_sq  = (cb ** 2).sum(-1).unsqueeze(0)
        dist   = res_sq - 2 * inner + cb_sq
        idx    = dist.argmin(-1)
        z_q_k  = cb[idx]

        stage_mses.append(F.mse_loss(z_q_k, residual).item())

        counts = torch.zeros(RVQ_CODEBOOK_SIZE, device=dev)
        counts.scatter_add_(0, idx, torch.ones(len(idx), device=dev))
        dead_codes.append(int((counts == 0).sum().item()))
        p = counts / counts.sum().clamp(min=1e-9)
        m = p > 0
        H = -(p[m] * p[m].log()).sum().item()
        perplexities.append(2 ** H)

        z_q_sum  = z_q_sum + z_q_k
        residual = residual - z_q_k

    final_mse   = F.mse_loss(z_q_sum, val_sample).item()
    final_align = F.cosine_similarity(z_q_sum, val_sample, dim=-1).mean().item()

    # MaxSim preservation: compare MaxSim(z_q) vs MaxSim(x) on same pairs
    # Sample 256 pairs for efficiency
    n_pairs = min(256, val_sample.shape[0] // 2)
    perm    = torch.randperm(val_sample.shape[0])
    anchors   = val_sample[perm[:n_pairs]]       # raw embeddings (queries)
    docs_raw  = val_sample[perm[n_pairs:2*n_pairs]]  # raw doc embeddings
    docs_q    = z_q_sum[perm[n_pairs:2*n_pairs]]     # quantized doc embeddings
    # Single-patch MaxSim (each row = one patch "document")
    maxsim_raw  = (anchors * docs_raw).sum(dim=-1).mean().item()
    maxsim_quant = (anchors * F.normalize(docs_q, dim=-1)).sum(dim=-1).mean().item()
    maxsim_ratio = maxsim_quant / max(abs(maxsim_raw), 1e-8)

    return {
        'perplexity_per_stage': [round(p, 1) for p in perplexities],
        'stage_recon_mse':      [round(m, 6) for m in stage_mses],
        'dead_codes_per_stage': dead_codes,
        'total_recon_mse':      round(final_mse, 6),
        'cosine_alignment':     round(final_align, 4),
        'maxsim_preservation':  round(maxsim_ratio, 4),
        'mean_perplexity':      float(np.mean(perplexities)),
        'min_perplexity':       float(np.min(perplexities)),
        'total_dead_codes':     sum(dead_codes),
    }


# ==============================================================================
# STEP 7: TRAIN BOTH PIPELINES AND SAVE TWO .pt FILES
# ==============================================================================

nq_levels = sorted(set([RVQ_N_QUANTIZERS_COARSE, RVQ_N_QUANTIZERS_FINE]))

# ── Shared val set for fair comparison ───────────────────────────────────────
val_size = min(8192, train_data.shape[0] // 5)
val_idx  = torch.randperm(train_data.shape[0])[:val_size]
val_data = train_data[val_idx]

# Storage for both approaches
all_codebooks_kmeans = {}
all_codebooks_spn    = {}
all_diag_kmeans      = {}
all_diag_spn         = {}


# ────────────────────────────────────────────────────────────────────────────
# PIPELINE A: KMeans + Ranking Losses
# ────────────────────────────────────────────────────────────────────────────
print("\n" + "█"*70)
print("PIPELINE A: KMeans Init + Ranking Losses (Baseline++)")
print("█"*70)

for NQ in nq_levels:
    print(f"\n{'─'*70}")
    print(f"  [KMeans] NQ={NQ}")
    print(f"{'─'*70}")
    t0 = time.perf_counter()
    model_km = train_rvq_kmeans(NQ, train_data, device, emb_dim)
    print(f"  Total elapsed: {time.perf_counter()-t0:.1f}s")

    cbs_km = extract_codebooks(model_km, NQ, device)
    all_codebooks_kmeans[NQ] = [cb.cpu() for cb in cbs_km]

    diag_km = codebook_diagnostics(cbs_km, val_data, device)
    all_diag_kmeans[NQ] = diag_km

    print(f"\n  [KMeans] Diagnostics NQ={NQ}:")
    print(f"    Alignment:          {diag_km['cosine_alignment']:.4f}")
    print(f"    MaxSim preservation:{diag_km['maxsim_preservation']:.4f}  (target >0.90)")
    print(f"    Recon MSE:          {diag_km['total_recon_mse']:.6f}")
    print(f"    Mean perplexity:    {diag_km['mean_perplexity']:.1f} / {RVQ_CODEBOOK_SIZE} "
          f"({diag_km['mean_perplexity']/RVQ_CODEBOOK_SIZE*100:.0f}% effective)")
    print(f"    Min  perplexity:    {diag_km['min_perplexity']:.1f}")
    print(f"    Dead codes:         {diag_km['total_dead_codes']} / {NQ * RVQ_CODEBOOK_SIZE}")

    del model_km; gc.collect(); torch.cuda.empty_cache()


# ────────────────────────────────────────────────────────────────────────────
# PIPELINE B: SPN Init + Ranking Losses
# ────────────────────────────────────────────────────────────────────────────
print("\n" + "█"*70)
print("PIPELINE B: SPN Init + Ranking Losses (Proposed)")
print("█"*70)

for NQ in nq_levels:
    print(f"\n{'─'*70}")
    print(f"  [SPN] NQ={NQ}")
    print(f"{'─'*70}")
    t0 = time.perf_counter()
    model_spn = train_rvq_spn(NQ, train_data, device, emb_dim)
    print(f"  Total elapsed: {time.perf_counter()-t0:.1f}s")

    cbs_spn = extract_codebooks(model_spn, NQ, device)
    all_codebooks_spn[NQ] = [cb.cpu() for cb in cbs_spn]

    diag_spn = codebook_diagnostics(cbs_spn, val_data, device)
    all_diag_spn[NQ] = diag_spn

    print(f"\n  [SPN] Diagnostics NQ={NQ}:")
    print(f"    Alignment:          {diag_spn['cosine_alignment']:.4f}")
    print(f"    MaxSim preservation:{diag_spn['maxsim_preservation']:.4f}  (target >0.90)")
    print(f"    Recon MSE:          {diag_spn['total_recon_mse']:.6f}")
    print(f"    Mean perplexity:    {diag_spn['mean_perplexity']:.1f} / {RVQ_CODEBOOK_SIZE} "
          f"({diag_spn['mean_perplexity']/RVQ_CODEBOOK_SIZE*100:.0f}% effective)")
    print(f"    Min  perplexity:    {diag_spn['min_perplexity']:.1f}")
    print(f"    Dead codes:         {diag_spn['total_dead_codes']} / {NQ * RVQ_CODEBOOK_SIZE}")

    del model_spn; gc.collect(); torch.cuda.empty_cache()


# ────────────────────────────────────────────────────────────────────────────
# HEAD-TO-HEAD COMPARISON TABLE
# ────────────────────────────────────────────────────────────────────────────
print("\n" + "="*70)
print("HEAD-TO-HEAD COMPARISON: KMeans++ vs SPN (same ranking losses)")
print("="*70)
print(f"  {'NQ':>4}  {'Method':>8}  {'Align':>6}  {'MaxSim':>7}  "
      f"{'MSE':>8}  {'Perp%':>6}  {'Dead':>5}")
print(f"  {'─'*55}")
for NQ in nq_levels:
    dk = all_diag_kmeans[NQ]
    ds = all_diag_spn[NQ]
    for method, d in [('KMeans', dk), ('SPN', ds)]:
        print(f"  {NQ:>4}  {method:>8}  "
              f"{d['cosine_alignment']:>6.4f}  "
              f"{d['maxsim_preservation']:>7.4f}  "
              f"{d['total_recon_mse']:>8.6f}  "
              f"{d['mean_perplexity']/RVQ_CODEBOOK_SIZE*100:>5.0f}%  "
              f"{d['total_dead_codes']:>5}")


# ==============================================================================
# STEP 8: SAVE TWO ARTIFACT FILES
# ==============================================================================
print("\n" + "="*70)
print("STEP 8: Save artifacts")
print("="*70)

# ── File 1: KMeans codebooks ─────────────────────────────────────────────────
torch.save({
    'codebooks':         all_codebooks_kmeans,
    'diagnostics':       all_diag_kmeans,
    'emb_dim':           emb_dim,
    'codebook_size':     RVQ_CODEBOOK_SIZE,
    'nq_coarse':         RVQ_N_QUANTIZERS_COARSE,
    'nq_fine':           RVQ_N_QUANTIZERS_FINE,
    'norm_filter':       TOP_K_NORM_FRAC,
    'n_train_pages':     len(layouts_df),
    'n_train_patches':   int(train_data.shape[0]),
    'dropout_p':         RVQ_DROPOUT_P,
    'train_datasets':    ALL_DATASETS,
    'init_method':       'KMeans',
    'rank_loss_weight':  RANK_LOSS_WEIGHT,
    'rank_margin':       RANK_MARGIN,
    'ot_loss_weight':    OT_LOSS_WEIGHT,
    'ot_sinkhorn_eps':   OT_SINKHORN_EPS,
    'loss_components':   ['recon', 'align', 'commit', 'rank_contrastive', 'sinkhorn_ot'],
}, CODEBOOK_SAVE_KMEANS)

sz_km = os.path.getsize(CODEBOOK_SAVE_KMEANS) / 1024**2
print(f"  [A] KMeans codebooks → {CODEBOOK_SAVE_KMEANS}  ({sz_km:.1f} MB)")

# ── File 2: SPN codebooks ─────────────────────────────────────────────────────
torch.save({
    'codebooks':         all_codebooks_spn,
    'diagnostics':       all_diag_spn,
    'emb_dim':           emb_dim,
    'codebook_size':     RVQ_CODEBOOK_SIZE,
    'nq_coarse':         RVQ_N_QUANTIZERS_COARSE,
    'nq_fine':           RVQ_N_QUANTIZERS_FINE,
    'norm_filter':       TOP_K_NORM_FRAC,
    'n_train_pages':     len(layouts_df),
    'n_train_patches':   int(train_data.shape[0]),
    'dropout_p':         RVQ_DROPOUT_P,
    'train_datasets':    ALL_DATASETS,
    'init_method':       'SPN',
    'spn_config': {
        'epochs':         SPN_EPOCHS,
        'hidden_dim':     SPN_HIDDEN_DIM,
        'temp_start':     SPN_TEMP_START,
        'temp_end':       SPN_TEMP_END,
        'entropy_weight': SPN_ENTROPY_WEIGHT,
        'batch_size':     SPN_BATCH_SIZE,
    },
    'rank_loss_weight':  RANK_LOSS_WEIGHT,
    'rank_margin':       RANK_MARGIN,
    'ot_loss_weight':    OT_LOSS_WEIGHT,
    'ot_sinkhorn_eps':   OT_SINKHORN_EPS,
    'loss_components':   ['recon', 'align', 'commit', 'rank_contrastive', 'sinkhorn_ot'],
}, CODEBOOK_SAVE_SPN)

sz_spn = os.path.getsize(CODEBOOK_SAVE_SPN) / 1024**2
print(f"  [B] SPN codebooks    → {CODEBOOK_SAVE_SPN}  ({sz_spn:.1f} MB)")

# ==============================================================================
# SUMMARY
# ==============================================================================
print("\n" + "="*70)
print("CELL 1 COMPLETE — Two codebook files saved")
print("="*70)
print(f"\n  File A: {CODEBOOK_SAVE_KMEANS}")
print(f"    Init:   KMeans (Lloyd's, 10 iters)")
print(f"    Train:  Dropout + Contrastive ranking (λ={RANK_LOSS_WEIGHT}) + Sinkhorn-OT (λ={OT_LOSS_WEIGHT})")
print(f"\n  File B: {CODEBOOK_SAVE_SPN}")
print(f"    Init:   SPN (per-stage, entropy-max, spherical)")
print(f"    Train:  Dropout + Contrastive ranking (λ={RANK_LOSS_WEIGHT}) + Sinkhorn-OT (λ={OT_LOSS_WEIGHT})")
print(f"\n  Shared losses:")
print(f"    L_total = L_recon + 0.5*L_align + 0.25*L_commit")
print(f"            + {RANK_LOSS_WEIGHT}*L_rank  [contrastive triplet, in-batch negatives]")
print(f"            + {OT_LOSS_WEIGHT}*L_OT    [Sinkhorn-Wasserstein, ε={OT_SINKHORN_EPS}]")
print(f"\n  Diagnostics summary:")
for NQ in nq_levels:
    dk, ds = all_diag_kmeans[NQ], all_diag_spn[NQ]
    print(f"  NQ={NQ:2d}  KMeans: align={dk['cosine_alignment']:.4f}  "
          f"maxsim={dk['maxsim_preservation']:.4f}  "
          f"dead={dk['total_dead_codes']}/{NQ*RVQ_CODEBOOK_SIZE}")
    print(f"        SPN:    align={ds['cosine_alignment']:.4f}  "
          f"maxsim={ds['maxsim_preservation']:.4f}  "
          f"dead={ds['total_dead_codes']}/{NQ*RVQ_CODEBOOK_SIZE}")
print(f"\n  Load in Cell 2:")
print(f"    kmeans_cbs = torch.load('{CODEBOOK_SAVE_KMEANS}')")
print(f"    spn_cbs    = torch.load('{CODEBOOK_SAVE_SPN}')")
print("="*70)